# Live exercise: a chart makeover, worked version

Below is a bad chart. Every number in it is right, and it still fails: it
does not say what it is for, and a reader cannot get the answer out of it.
The exercise is to rebuild it one decision at a time, and to be able to say
**why** at every step.

Try each step in the exercise notebook before reading it here.

In [ ]:
import os
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd

here = Path.cwd()
while not (here / "data" / "clean" / "plays.csv").exists() and here != here.parent:
    here = here.parent
os.chdir(here)
os.makedirs("output", exist_ok=True)

plays = pd.read_csv("data/clean/plays.csv")
plays["year"] = plays["played_at"].str[:4].astype(int)

per_year = plays.pivot_table(index="genre", columns="year", values="play_id", aggfunc="count")
per_year

## The chart you were given

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
per_year.plot(ax=ax, marker="s", linewidth=3, color=["red", "green"])
ax.set_ylim(55, 290)
ax.set_title("Genre data")
ax.grid(True, linewidth=1.5)
plt.show()

## Step 0: what is wrong with it?

Say it out loud before touching the code. The list we found:

1. **no question**: "Genre data" is a topic, so the reader cannot tell
   what to look for
2. **a line between categories**: Afrobeat does not lead to Electronic, so
   the slope between them means nothing
3. **alphabetical order**: the reader has to hunt for the biggest and
   smallest
4. **the axis starts at 55**, so small genres look like nothing
5. **red against green**, the one pair that one man in twelve cannot tell
   apart
6. **the answer needs arithmetic**: to see how a genre changed, the reader
   has to subtract one line from the other, in their head, eight times
7. **clutter**: a heavy grid, square markers, no axis labels, no units

Number 6 is the big one, and you only see it once you know the question.

## Step 1: the question, in one sentence

**Which genres grew from 2024 to 2025, and which shrank?**

## Step 2: the number that answers it

The question is about **change**, so the chart should show the change, not
the two years. Compute it, in plays and as a percentage of 2024.

In [ ]:
per_year["change"] = per_year[2025] - per_year[2024]
per_year["percent"] = per_year["change"] / per_year[2024] * 100
per_year = per_year.sort_values("percent")
per_year.round(1)

**Plays or percent?** Electronic lost 11 plays and Latin lost 11 plays.
That sounds equal, but Electronic had 279 to lose and Latin had 98: 3.9%
against 11.2%. A percentage compares each genre with its own past, which is
the fair comparison here. Out of what? Out of that genre's 2024 plays. Keep
the plays in the labels, so the reader sees how small some groups are.

In [ ]:
assert per_year.loc["Electronic", "change"] == per_year.loc["Latin", "change"] == -11
assert round(per_year.loc["Electronic", "percent"], 1) == -3.9
assert round(per_year.loc["Latin", "percent"], 1) == -11.2

## Step 3: the right kind of chart

Comparing categories: bars. Horizontal bars, because some changes are
negative and the eye reads "left is down, right is up" easily, and because
genre names fit better sideways. Already sorted, so the extremes are at the
ends.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(per_year.index, per_year["percent"])
plt.show()

## Step 4: zero is the line that matters

Bars from zero, which `barh` does by itself as long as nobody sets the
limits. With positive and negative bars, the zero line is what the reader
measures from, so draw it.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.barh(per_year.index, per_year["percent"])
ax.axvline(0, color="black", linewidth=1)
plt.show()

## Step 5: colour that means something

Colour now has a job: up or down. Blue and orange from Okabe-Ito, not
green and red. The colours are a help, not the message: the sign is also in
the labels.

In [ ]:
UP, DOWN = "#0072b2", "#e69f00"
colours = [UP if change > 0 else DOWN for change in per_year["change"]]

fig, ax = plt.subplots(figsize=(8, 4.5))
bars = ax.barh(per_year.index, per_year["percent"], color=colours)
ax.axvline(0, color="black", linewidth=1)
plt.show()

## Step 6: numbers a person would say

Each label has the percentage with `:+.1f`, which is `:.1f` with a sign
always shown, and the change in plays with `:+d`. `zip` walks the two
columns side by side.

In [ ]:
labels = [f"{percent:+.1f}% ({change:+d})"
          for percent, change in zip(per_year["percent"], per_year["change"])]
print(labels)

assert labels[0] == "-14.4% (-34)"
assert labels[-1] == "+16.9% (+14)"

## Step 7: a title that gives the answer

Computed from the data, so it cannot drift out of date.

In [ ]:
grew = per_year[per_year["change"] > 0].index.tolist()
fell_most = per_year.index[0]
grew_most = per_year.index[-1]

title = (f"Only {grew[1]} and {grew[0]} grew in 2025; "
         f"{fell_most} fell most, {per_year.loc[fell_most, 'percent']:.1f}%")
print(title)

assert grew == ["Jazz", "Afrobeat"]
assert (fell_most, grew_most) == ("Folk", "Afrobeat")

## Step 8: take away what says nothing, and save

No grid, no top or right border, no legend (there is only one series now),
and an axis label that says what the numbers are. Room on both sides so the
labels fit. Save before showing.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(per_year.index, per_year["percent"], color=colours)
ax.bar_label(bars, labels=labels, padding=4)
ax.axvline(0, color="black", linewidth=1)

ax.set_title(title)
ax.set_xlabel("change in plays from 2024 to 2025 (%)")
ax.set_xlim(-24, 26)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.savefig("output/genre_change.png", dpi=150, bbox_inches="tight")
plt.show()

Compare it with the chart at the top. Nothing about the data changed. The
reader now gets the answer from the title, checks it in two seconds against
the bars, and finds the size of every change written next to it.

## Step 9: what this chart does not show

It shows each genre against its own past. It does not show that **all**
listening fell:

In [ ]:
total_2024, total_2025 = per_year[2024].sum(), per_year[2025].sum()
overall = (total_2025 - total_2024) / total_2024 * 100
print(f"all plays: {total_2024} in 2024, {total_2025} in 2025, {overall:.1f}%")

assert (total_2024, total_2025) == (1122, 1061)
assert round(overall, 1) == -5.4

Down 5.4% overall. So Electronic, down 3.9%, and Indie, down 4.0%, fell
**less** than listening as a whole: their share of the log went up. "Shrank"
in plays and "shrank" in share are different claims. The chart supports the
first one only.

## Step 10, if there is time: a reference line changes the story

Add the overall change as a dashed line. Every bar to the right of it
gained share; every bar to the left lost it.

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4.5))
bars = ax.barh(per_year.index, per_year["percent"], color=colours)
ax.bar_label(bars, labels=labels, padding=4)
ax.axvline(0, color="black", linewidth=1)
ax.axvline(overall, color="#6b7280", linestyle="--", linewidth=1)
ax.annotate(f"all listening: {overall:.1f}%", xy=(overall, 7.4), xytext=(-6, 0),
            textcoords="offset points", ha="right", color="#6b7280")

gained_share = per_year[per_year["percent"] > overall].index.tolist()
ax.set_title(f"{len(gained_share)} of 8 genres fell less than listening as a whole")
ax.set_xlabel("change in plays from 2024 to 2025 (%)")
ax.set_xlim(-24, 26)
ax.spines[["top", "right", "left"]].set_visible(False)
ax.tick_params(axis="y", length=0)

fig.savefig("output/genre_change_with_reference.png", dpi=150, bbox_inches="tight")
plt.show()

assert gained_share == ["Indie", "Electronic", "Jazz", "Afrobeat"]

Same bars, one dashed line, and a different and more accurate title: four
of the eight genres did better than listening as a whole. Which of the two
titles is right depends on which question you were asked, which is why the
question comes first.